# Pipeline e testes de sanidade da DeepResUNet

**Objetivo:** verificar visualmente e numericamente o pipeline inteiro antes de qualquer baseline completo.

Os resultados deste notebook são **diagnósticos técnicos**, não desempenho científico. Somente treino e validação são permitidos; teste e FUSeg ficam fechados.

## Mapa da execução

`dados → DeepResUNet → logits → BCE+Dice → atualização`

`logits → sigmoid → probabilidade → threshold 0,5 → máscara → métricas`

Primeiro a rede tentará memorizar quatro imagens. Depois faremos uma época curta em 16 imagens de treino e 8 de validação e testaremos o checkpoint.

## 1. Montar o Drive e localizar o projeto

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Execução fora do Colab.")

candidates = [
    Path("/content/drive/MyDrive/TCC/Projeto_TCC_Lucas"),
    Path("/content/drive/MyDrive/Projeto_TCC_Lucas"),
]
PROJECT_ROOT = next((path for path in candidates if path.is_dir()), Path.cwd())
if not (PROJECT_ROOT / "configs/pipeline_sanity.yaml").is_file():
    raise FileNotFoundError("Projeto ou configuração da verificação do pipeline não encontrados.")
os.chdir(PROJECT_ROOT)
print("Projeto:", PROJECT_ROOT)

## 2. Instalar as dependências registradas

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()

## 3. Copiar os dados para o disco temporário

O Drive guarda os arquivos permanentes. A cópia temporária acelera a leitura sem alterar os dados originais.

In [ ]:
import shutil

DATA_ROOT = Path("/content/tcc_data") if Path("/content").is_dir() else PROJECT_ROOT
if DATA_ROOT != PROJECT_ROOT:
    shutil.copytree(PROJECT_ROOT / "Dataset", DATA_ROOT / "Dataset", dirs_exist_ok=True)
print("Raiz usada para leitura:", DATA_ROOT)

## 4. Carregar configuração, bibliotecas e fixar a seed

In [ ]:
import inspect, json, time, yaml
import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image
from IPython.display import display

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from tcc_segmentation.data import DevelopmentDataset, ResizePad, create_development_dataloader
from tcc_segmentation.pipeline_sanity import (
    batch_diagnostics,
    collect_predictions,
    deterministic_prefix_indices,
    indices_for_canonical_ids,
    prediction_max_abs_difference,
    run_overfit_sanity,
    run_smoke_sanity,
    save_before_after_panel,
    save_history_csv,
    write_json,
)
from tcc_segmentation.models import DeepResUNet, build_segmentation_model, count_parameters
from tcc_segmentation.training import (
    BCEDiceLoss,
    load_training_checkpoint,
    save_training_checkpoint,
    seed_everything,
)

CONFIG = yaml.safe_load((PROJECT_ROOT / "configs/pipeline_sanity.yaml").read_text(encoding="utf-8"))
seed_everything(int(CONFIG["seed"]))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_ID = time.strftime("day4_%Y%m%dT%H%M%SZ", time.gmtime())
OUTPUT_DIR = PROJECT_ROOT / CONFIG["runtime"]["output_root"] / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print("Device:", DEVICE)
print("Saídas:", OUTPUT_DIR)
display(CONFIG)

## 5. Criar datasets permitidos

A classe abaixo rejeita manifestos protegidos e confere o hash dos manifestos de treino e validação. Nenhum caminho de teste aparece nesta configuração.

In [ ]:
transform = ResizePad(height=CONFIG["data"]["height"], width=CONFIG["data"]["width"])
train_dataset = DevelopmentDataset(
    PROJECT_ROOT / CONFIG["data"]["train_manifest"],
    DATA_ROOT,
    "train",
    transform=transform,
    mask_threshold=CONFIG["data"]["mask_threshold"],
)
validation_dataset = DevelopmentDataset(
    PROJECT_ROOT / CONFIG["data"]["validation_manifest"],
    DATA_ROOT,
    "validation",
    transform=transform,
    mask_threshold=CONFIG["data"]["mask_threshold"],
)
print(
    {
        "treino": len(train_dataset),
        "validação": len(validation_dataset),
        "teste_acessado": False,
        "FUSeg_acessado": False,
    }
)

## 6. Visualizar entradas e referências antes do treinamento

In [ ]:
ids = CONFIG["overfit"]["canonical_ids"]
indices = indices_for_canonical_ids(train_dataset, ids)
overfit_dataset = torch.utils.data.Subset(train_dataset, indices)
overfit_loader = create_development_dataloader(
    overfit_dataset,
    batch_size=CONFIG["overfit"]["batch_size"],
    shuffle=False,
    seed=CONFIG["seed"],
    num_workers=0,
)
batch = next(iter(overfit_loader))
diagnostics = batch_diagnostics(batch)
display(diagnostics)

fig, axes = plt.subplots(4, 2, figsize=(10, 14))
for row, sample_index in enumerate(range(4)):
    sample = overfit_dataset[sample_index]
    axes[row, 0].imshow(sample["image"].transpose(1, 2, 0))
    axes[row, 0].set_title(sample["canonical_id"] + " — imagem")
    axes[row, 1].imshow(sample["mask"][0], cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("máscara canônica")
    axes[row, 0].axis("off")
    axes[row, 1].axis("off")
plt.tight_layout()
plt.show()

## 7. Conferir a arquitetura central

A sigmoid foi retirada apenas da saída final. Os attention gates continuam usando sigmoid internamente para calcular seus coeficientes de atenção.

In [ ]:
print(inspect.getsource(DeepResUNet))
model = build_segmentation_model(CONFIG["model"]).to(DEVICE)
print("Parâmetros:", count_parameters(model))

## 8. Fazer uma passagem e observar logits, probabilidades e loss

In [ ]:
images = batch["image"].to(DEVICE)
masks = batch["mask"].to(DEVICE)
model.eval()
with torch.inference_mode():
    logits = model(images).float()
    probabilities = torch.sigmoid(logits)
criterion = BCEDiceLoss(
    CONFIG["loss"]["bce_weight"], CONFIG["loss"]["dice_weight"], CONFIG["loss"]["smooth"]
)
components = criterion.components(logits, masks)
display(
    {
        "shape_logits": list(logits.shape),
        "shape_máscaras": list(masks.shape),
        "logit_min": float(logits.min()),
        "logit_max": float(logits.max()),
        "probabilidade_min": float(probabilities.min()),
        "probabilidade_max": float(probabilities.max()),
        "loss_total": float(components["total"]),
        "bce": float(components["bce"]),
        "dice_loss": float(components["dice_loss"]),
        "tudo_finito": bool(torch.isfinite(logits).all()),
    }
)

## 9. Registrar as predições antes e executar o overfit pequeno

A execução pode parar antes de 300 épocas se a loss cair pelo menos 80% e o Dice técnico médio atingir 0,90. Isso é um gate de depuração, não seleção de modelo.

In [ ]:
seed_everything(CONFIG["seed"])
model = build_segmentation_model(CONFIG["model"]).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["overfit"]["learning_rate"])
before = collect_predictions(model, overfit_loader, DEVICE)
history, overfit_accepted = run_overfit_sanity(
    model=model,
    loader=overfit_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=DEVICE,
    epochs=CONFIG["overfit"]["epochs"],
    threshold=0.5,
    min_loss_reduction_fraction=CONFIG["overfit"]["early_acceptance"][
        "min_loss_reduction_fraction"
    ],
    min_macro_dice=CONFIG["overfit"]["early_acceptance"]["min_macro_dice"],
)
after = collect_predictions(model, overfit_loader, DEVICE)
history_df = pd.DataFrame(history)
display(history_df.tail())
print("Gate do overfit aprovado:", overfit_accepted)

## 10. Curvas e comparação visual antes/depois

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["loss"])
axes[0].set(title="Loss — overfit pequeno", xlabel="Época", ylabel="Loss")
axes[1].plot(history_df["epoch"], history_df["macro_dice_at_0_5"])
axes[1].set(title="Dice técnico em threshold 0,5", xlabel="Época", ylabel="Dice")
plt.tight_layout()
plt.show()
panel_path = save_before_after_panel(before, after, OUTPUT_DIR / "overfit_before_after.png", 0.5)
display(Image.open(panel_path))
save_history_csv(history, OUTPUT_DIR / "overfit_history.csv")

## 11. Smoke test curto com modelo reinicializado

O modelo é recriado para que a memorização anterior não contamine este teste. As métricas abaixo apenas confirmam que treino e validação executam.

In [ ]:
smoke = CONFIG["smoke"]
smoke_train = torch.utils.data.Subset(
    train_dataset, deterministic_prefix_indices(len(train_dataset), smoke["train_samples"])
)
smoke_validation = torch.utils.data.Subset(
    validation_dataset,
    deterministic_prefix_indices(len(validation_dataset), smoke["validation_samples"]),
)
smoke_train_loader = create_development_dataloader(
    smoke_train, smoke["batch_size"], True, CONFIG["seed"], 0
)
smoke_validation_loader = create_development_dataloader(
    smoke_validation, smoke["batch_size"], False, CONFIG["seed"], 0
)
seed_everything(CONFIG["seed"])
smoke_model = build_segmentation_model(CONFIG["model"]).to(DEVICE)
smoke_optimizer = torch.optim.Adam(smoke_model.parameters(), lr=smoke["learning_rate"])
smoke_result = run_smoke_sanity(
    model=smoke_model,
    train_loader=smoke_train_loader,
    validation_loader=smoke_validation_loader,
    criterion=criterion,
    optimizer=smoke_optimizer,
    device=DEVICE,
    threshold=0.5,
)
display(
    {
        "resultado_científico": False,
        "loss_treino": smoke_result["train"]["loss"],
        "loss_validação": smoke_result["validation"]["loss"],
        "métricas_técnicas_validação": {
            k: v["mean"]
            for k, v in smoke_result["validation"]["metrics"].items()
            if isinstance(v, dict)
        },
    }
)

## 12. Salvar, recarregar e comparar o checkpoint

In [ ]:
predictions_before_reload = collect_predictions(smoke_model, smoke_validation_loader, DEVICE)
split_summary = json.loads((PROJECT_ROOT / "manifests/splits/split_summary.json").read_text())
checkpoint_path = OUTPUT_DIR / "smoke_checkpoint.pt"
save_training_checkpoint(
    checkpoint_path,
    kind="last",
    epoch=1,
    best_score=smoke_result["validation"]["metrics"]["dice"]["mean"],
    model=smoke_model,
    optimizer=smoke_optimizer,
    config=CONFIG,
    run_id=RUN_ID,
    manifest_hashes={k: split_summary["manifest_hashes"][k] for k in ("train", "validation")},
    extra={"scientific_result": False, "protected_partitions_accessed": False},
)
reloaded_model = build_segmentation_model(CONFIG["model"]).to(DEVICE)
reloaded_optimizer = torch.optim.Adam(reloaded_model.parameters(), lr=smoke["learning_rate"])
load_training_checkpoint(
    checkpoint_path,
    model=reloaded_model,
    optimizer=reloaded_optimizer,
    map_location=DEVICE,
    restore_rng=False,
)
predictions_after_reload = collect_predictions(reloaded_model, smoke_validation_loader, DEVICE)
roundtrip_error = prediction_max_abs_difference(predictions_before_reload, predictions_after_reload)
print("Maior diferença absoluta entre logits:", roundtrip_error)
print("Checkpoint reproduz exatamente:", roundtrip_error <= 1e-6)

## 13. Gerar o relatório técnico final

In [ ]:
report = {
    "status": "passed" if overfit_accepted and roundtrip_error <= 1e-6 else "review",
    "scientific_result": False,
    "run_id": RUN_ID,
    "device": str(DEVICE),
    "model_parameters": count_parameters(smoke_model),
    "overfit": {
        "canonical_ids": ids,
        "epochs_executed": len(history),
        "accepted": overfit_accepted,
        "first": history[0],
        "last": history[-1],
    },
    "smoke": {
        "train_samples": len(smoke_train),
        "validation_samples": len(smoke_validation),
        "checkpoint_roundtrip_max_abs_logit_error": roundtrip_error,
    },
    "partitions_accessed": ["train", "validation"],
    "protected_partitions_accessed": False,
}
write_json(diagnostics, OUTPUT_DIR / "batch_diagnostics.json")
write_json(smoke_result, OUTPUT_DIR / "smoke_summary.json")
write_json(report, OUTPUT_DIR / "day4_sanity_report.json")
display(report)
print("Arquivos salvos em:", OUTPUT_DIR)

## Como interpretar

- `passed`: o pipeline está tecnicamente apto a avançar para o baseline.
- `review`: a verificação requer inspeção das curvas, do painel, dos gradientes ou do checkpoint.

Em nenhum caso as métricas deste notebook representam desempenho final da rede. Arquive `day4_sanity_report.json`, `overfit_history.csv`, `overfit_before_after.png` e `smoke_summary.json` para a auditoria da execução.